In [0]:
# Install packages needed by this workflow
%pip install pydeequ==1.4.0 openpyxl fuzzywuzzy python-Levenshtein

In [0]:
dbutils.library.restartPython()

In [0]:
import os
os.environ["SPARK_VERSION"] = spark.version

import numpy as np
import pandas as pd
import string
import json
import pydeequ

from openpyxl.utils import get_column_letter, column_index_from_string
from fuzzywuzzy import process, fuzz
from collections import defaultdict
from openpyxl import load_workbook
from shutil import copyfile
from pydeequ.checks import *
from pydeequ.verification import *
from pydeequ.repository import *
from pydeequ.analyzers import *
from openpyxl.utils.dataframe import dataframe_to_rows

In [0]:
%run ./Triage_Functions

In [0]:
#Get file paths and client_key from Wrapper notebook
for k in ["wc_loss_run_path", "liab_loss_run_path", "ttd_path", "ptr_path", "wc_ptr_path", "liab_ptr_path", "client_key"]:
    dbutils.widgets.text(k, "")

wc_loss_run_path   = dbutils.widgets.get("wc_loss_run_path")
liab_loss_run_path = dbutils.widgets.get("liab_loss_run_path")
ttd_path           = dbutils.widgets.get("ttd_path")
ptr_path           = dbutils.widgets.get("ptr_path")       # unlabeled PTR - assigned to WC or LIAB in load_client_sources
wc_ptr_path        = dbutils.widgets.get("wc_ptr_path")
liab_ptr_path      = dbutils.widgets.get("liab_ptr_path")
client_key         = dbutils.widgets.get("client_key")

In [0]:
# Define list of possible TPA selections. All_TPA is default
dbutils.widgets.dropdown("TPA", "All_TPA", ["CorVel", "CCMSI", "GB", "Helmsman", "ESIS", "Broadspire", "Sedgwick", "Sentry", "Travelers", "Zurich", "Other", "All_TPA"])

In [0]:
# Get parameters from job run configuration
chosen_tpa = dbutils.widgets.get("TPA")
tpa_path = '/Volumes/agrcnaanalytics_workbench/triage/triage_loss_run/mapping/' + chosen_tpa + '_Mapping_Dict.json'
print(tpa_path)
loss_run_mapping_dict = read_json_to_dict(tpa_path)
ttd_mapping_dict = read_json_to_dict('/Volumes/agrcnaanalytics_workbench/triage/triage_loss_run/mapping/TTD_master_dict.json')
ptr_mapping_dict = read_json_to_dict('/Volumes/agrcnaanalytics_workbench/triage/triage_loss_run/mapping/PTR_master_dict.json')

In [0]:
#Call load_client_sources function to find sheets containing project data and load them into memory
#Requires a WC and/or LIAB loss run. TTD/PTR are taken from standalone files if provided, otherwise searched for within the loss run workbook(s)
df_loss_run_wc, df_loss_run_liab, df_ttd, df_wc_ptr, df_liab_ptr = load_client_sources(
   wc_loss_run_path=wc_loss_run_path,
   liab_loss_run_path=liab_loss_run_path,
   ttd_path=ttd_path,
   ptr_path=ptr_path,
   wc_ptr_path=wc_ptr_path,
   liab_ptr_path=liab_ptr_path
)
has_wc = df_loss_run_wc is not None
print("Client:", client_key)
print("WC loaded:", has_wc)
print("LIAB loaded:", df_loss_run_liab is not None)
print("TTD loaded:", df_ttd is not None)
print("WC PTR loaded:", df_wc_ptr is not None)
print("LIAB PTR loaded:", df_liab_ptr is not None)

In [0]:
# Map WC Loss Run. Occupation is split out because it goes in the Hierarchy section of the triage form rather than the main field list
if has_wc:
    mapping_df_wc = triage_mapper(df_loss_run_wc, wc_target_fields, loss_run_mapping_dict)
    occupation_mask = mapping_df_wc['Field Name'] == 'Occupation'
    occupation_column = mapping_df_wc.loc[occupation_mask, 'Matched Column Letter'].iloc[0]
    mapping_df_wc = mapping_df_wc.loc[~occupation_mask].reset_index(drop=True)

In [0]:
# Map LIAB Loss Run, TTD, WC PTR and LIAB PTR
if df_loss_run_liab is not None:
    mapping_df_liab = triage_mapper(df_loss_run_liab, liab_target_fields, loss_run_mapping_dict)
if df_ttd is not None:
    mapping_df_ttd = triage_mapper(df_ttd, ttd_target_fields, ttd_mapping_dict)
if df_wc_ptr is not None:
    mapping_df_wc_ptr = triage_mapper(df_wc_ptr, wc_ptr_target_fields, ptr_mapping_dict)
if df_liab_ptr is not None:
    mapping_df_liab_ptr = triage_mapper(df_liab_ptr, liab_ptr_target_fields, ptr_mapping_dict)

In [0]:
#Copy DQ Assessment template into cluster memory and assign sheets to variables (WC Loss Run only - skipped if no WC loss run)
if has_wc:
    copyfile('/Volumes/agrcnaanalytics_workbench/triage/triage_loss_run/DQ_Assessment_Template.xlsx','/local_disk0/tmp/DQ_Assessment_Template.xlsx')
    dq_wb = load_workbook('/Volumes/agrcnaanalytics_workbench/triage/triage_loss_run/DQ_Assessment_Template.xlsx')
    dq_checks_ws = dq_wb['Claim-Level Checks']
    dq_summary_ws = dq_wb['Summary']
    dq_completeness_ws = dq_wb['Completeness Checks']
    dq_hierarchy_ws = dq_wb['Hierarchy Fields']

In [0]:
#Rename columns based on mapping dictionary (This function does not utilize LLM second-pass) (WC Loss Run only - skipped if no WC loss run)
if has_wc:
    df_loss_run_wc_renamed = column_matcher(df_loss_run_wc, loss_run_mapping_dict)

In [0]:
#Convert pandas df to spark (WC Loss Run only - skipped if no WC loss run)
if has_wc:
    renamed_loss_run_spark = spark.createDataFrame(df_loss_run_wc_renamed)

In [0]:
#Define hierarchy and metric fields
hierarchy_fields = ['Reporting Unit','Reporting Unit Number','Unit Name','Location','District','Jurisdiction State','Structure Level Name 01','Structure Level Name 02','Structure Level Name 03','Structure Level Name 04','Structure Level Name 05','Structure Level Name 06','Structure Level Name 07','Structure Level Name 08','Structure Level Name 09','Structure Level Name 10']
metric_field = 'Net Total Incurred'

In [0]:
# Run hierarchy profiler and append results (WC Loss Run only - skipped if no WC loss run)
if has_wc:
    profiling_output = hierarchy_profiler(renamed_loss_run_spark, hierarchy_fields, metric_field)
    profiling_output_pd = profiling_output.toPandas()
    for r in dataframe_to_rows(profiling_output_pd, index=False, header=False):
        dq_hierarchy_ws.append(r)

In [0]:
# Run DQ checks and append results (WC Loss Run only - skipped if no WC loss run)
if has_wc:
    dq_df = run_dq_checks(renamed_loss_run_spark)
    summary_df = dq_summary(dq_df, claim_id_col="Claim number")
    dq_df_pd = dq_df.toPandas()
    for r in dataframe_to_rows(dq_df_pd, index=False, header=True):
        dq_checks_ws.append(r)
    summary_df_pd = summary_df.toPandas()
    summary_df_pd = summary_df_pd.drop(columns=["unmapped_or_duplicate_pct"])
    for r in dataframe_to_rows(summary_df_pd, index=False, header=False):
        dq_summary_ws.append(r)

In [0]:
# Run completeness check and append results (WC Loss Run only - skipped if no WC loss run)
if has_wc:
    completeness_df = completeness_checks(renamed_loss_run_spark)
    completeness_df_pd = completeness_df.toPandas()
    for r in dataframe_to_rows(completeness_df_pd, index=False, header=False):
        dq_completeness_ws.append(r)
    # Save DQ WB
    dq_wb.save('/local_disk0/tmp/DQ_Assessment_Template.xlsx')

In [0]:
# Consultant Triage form template (Consultant_Triage_Template_New.xlsx must be uploaded to this Volume path)
TRIAGE_TEMPLATE_PATH = '/Volumes/agrcnaanalytics_workbench/triage/triage_loss_run/Consultant_Triage_Template_New.xlsx'
LOCAL_TRIAGE_PATH    = '/local_disk0/tmp/Consultant_Triage_Template.xlsx'

In [0]:
# Read in Triage form template and restore drop-downs that openpyxl drops on load
triage_wb = load_workbook(TRIAGE_TEMPLATE_PATH)
print("Restored drop-downs:", restore_x14_data_validations(TRIAGE_TEMPLATE_PATH, triage_wb))
triage_ws_wc       = triage_wb['WC Loss Run']
triage_ws_liab     = triage_wb['Liability Loss Run - ALGLPL ']
triage_ws_ttd      = triage_wb['WC TTD Report']
triage_ws_wc_ptr   = triage_wb['WC Payment Transaction Report']
triage_ws_liab_ptr = triage_wb['Liability Payment Trans Report']

In [0]:
# Insert mapped column letters into Triage form (row/cell positions below match Consultant_Triage_Template_New.xlsx)
# Only mapped fields are written - client details, Industry Group, NAICS, dates etc. are completed manually
start_row_wc      = 27      # WC Loss Run: 'Claim number' row
occupation_cell   = 'C55'   # WC Loss Run: 'Hierarchy Level 3 - Occupation'
start_row_liab    = 21      # Liability Loss Run: 'Claim number' row
start_row_ttd_ptr = 11      # TTD / PTR sheets: first field row

if has_wc:
    write_mapping_to_sheet(triage_ws_wc, mapping_df_wc, start_row_wc)
    triage_ws_wc[occupation_cell] = occupation_column
if df_loss_run_liab is not None:
    write_mapping_to_sheet(triage_ws_liab, mapping_df_liab, start_row_liab)
if df_ttd is not None:
    write_mapping_to_sheet(triage_ws_ttd, mapping_df_ttd, start_row_ttd_ptr)
if df_wc_ptr is not None:
    write_mapping_to_sheet(triage_ws_wc_ptr, mapping_df_wc_ptr, start_row_ttd_ptr)
if df_liab_ptr is not None:
    write_mapping_to_sheet(triage_ws_liab_ptr, mapping_df_liab_ptr, start_row_ttd_ptr)

# Remove sheets for data sets that were not included in the input files (e.g. WC_Loss_Run + TTD + WC_PTR -> only those 3 sheets remain)
sheet_included = [
    (triage_ws_wc,       has_wc),
    (triage_ws_liab,     df_loss_run_liab is not None),
    (triage_ws_ttd,      df_ttd is not None),
    (triage_ws_wc_ptr,   df_wc_ptr is not None),
    (triage_ws_liab_ptr, df_liab_ptr is not None),
]
remove_unused_sheets(triage_wb, [ws for ws, included in sheet_included if not included])

triage_wb.save(LOCAL_TRIAGE_PATH)

In [0]:
client_string = client_key.capitalize()

In [0]:
# Copying finalized triage form to output folder (DQ Assessment is only produced when a WC loss run is present)
copyfile(LOCAL_TRIAGE_PATH,'/Volumes/agrcnaanalytics_workbench/triage/triage_loss_run/output/' + client_string + '_Consultant_Triage_Draft.xlsx')
if has_wc:
    copyfile('/local_disk0/tmp/DQ_Assessment_Template.xlsx','/Volumes/agrcnaanalytics_workbench/triage/triage_loss_run/output/' + client_string + '_DQ_Assessment.xlsx')